# Paper 7: Graph Consequences of DNA Serialization — Disruption Order Follows Network Degree

**Reproducibility notebook** for McCarthy (2026), Paper 7.

Tests whether network position (hub vs leaf, derived from STRING protein-protein
interaction degree) predicts mutation severity within coupling channels. Key finding:
hub mutations in bottleneck channels (CellCycle, PI3K/Growth) carry worse prognosis
(HR 1.30–1.47), while the parallel-redundant DDR channel shows no hub/leaf effect.

**Data sources:**
- MSK-MET 2021 — MetTropism Cohort (Nguyen et al. 2022), fetched from cBioPortal API
- STRING v12 protein-protein interaction network (species 9606, score ≥ 700)

Runtime: ~5–10 minutes (API-bound).

In [ ]:
# Cell 1: Setup
import os
import json
import time as _time
import warnings
import requests
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
%matplotlib inline

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
print("Setup complete.")

In [ ]:
# cBioPortal API helpers
import time as _time

BASE_URL = "https://www.cbioportal.org/api"
HEADERS = {"Accept": "application/json"}


def api_get(path, params=None, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.get(f"{BASE_URL}{path}", headers=HEADERS, params=params, timeout=60)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                _time.sleep(5 * (attempt + 1))
            else:
                raise


def api_post(path, body, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.post(
                f"{BASE_URL}{path}", json=body,
                headers={**HEADERS, "Content-Type": "application/json"}, timeout=60
            )
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                _time.sleep(5 * (attempt + 1))
            else:
                raise

In [ ]:
# Cell 3: Channel mapping — 122 genes across 8 coupling channels

CHANNEL_MAP = {
    # DDR — DNA Damage Response
    "ATM": "DDR", "ATR": "DDR", "BRCA1": "DDR", "BRCA2": "DDR",
    "PALB2": "DDR", "RAD51C": "DDR", "RAD51D": "DDR", "RAD51B": "DDR",
    "CHEK1": "DDR", "CHEK2": "DDR", "BAP1": "DDR", "BARD1": "DDR",
    "FANCA": "DDR", "FANCC": "DDR", "FANCD2": "DDR", "MLH1": "DDR",
    "MSH2": "DDR", "MSH6": "DDR", "PMS2": "DDR", "POLE": "DDR",
    "POLD1": "DDR",

    # CellCycle — Cell Cycle / Apoptosis
    "TP53": "CellCycle", "RB1": "CellCycle", "CDKN1A": "CellCycle", "CDKN1B": "CellCycle",
    "CDKN2A": "CellCycle", "CDKN2B": "CellCycle", "CDK4": "CellCycle", "CDK6": "CellCycle",
    "CCND1": "CellCycle", "CCNE1": "CellCycle", "MDM2": "CellCycle", "MDM4": "CellCycle",
    "MYC": "CellCycle", "MYCN": "CellCycle",

    # PI3K_Growth — PI3K / Growth Signaling
    "PIK3CA": "PI3K_Growth", "PIK3R1": "PI3K_Growth", "PTEN": "PI3K_Growth", "AKT1": "PI3K_Growth",
    "AKT2": "PI3K_Growth", "AKT3": "PI3K_Growth", "MTOR": "PI3K_Growth", "KRAS": "PI3K_Growth",
    "NRAS": "PI3K_Growth", "HRAS": "PI3K_Growth", "BRAF": "PI3K_Growth", "RAF1": "PI3K_Growth",
    "MAP2K1": "PI3K_Growth", "MAP2K2": "PI3K_Growth", "MAP3K1": "PI3K_Growth", "MAP3K13": "PI3K_Growth",
    "ERBB2": "PI3K_Growth", "ERBB3": "PI3K_Growth", "EGFR": "PI3K_Growth", "FGFR1": "PI3K_Growth",
    "FGFR2": "PI3K_Growth", "FGFR3": "PI3K_Growth", "IGF1R": "PI3K_Growth", "MET": "PI3K_Growth",
    "NF1": "PI3K_Growth", "NF2": "PI3K_Growth", "TSC1": "PI3K_Growth", "TSC2": "PI3K_Growth",
    "STK11": "PI3K_Growth",

    # Endocrine — Endocrine / Hormone Receptor
    "ESR1": "Endocrine", "ESR2": "Endocrine", "PGR": "Endocrine", "AR": "Endocrine",
    "FOXA1": "Endocrine", "GATA3": "Endocrine",

    # Immune — Immune Surveillance
    "B2M": "Immune", "HLA-A": "Immune", "HLA-B": "Immune", "HLA-C": "Immune",
    "JAK1": "Immune", "JAK2": "Immune", "STAT1": "Immune", "CD274": "Immune",
    "PDCD1LG2": "Immune", "CTLA4": "Immune",

    # TissueArchitecture — Tissue Architecture
    "CDH1": "TissueArchitecture", "CDH2": "TissueArchitecture", "CTNNB1": "TissueArchitecture", "APC": "TissueArchitecture",
    "AXIN1": "TissueArchitecture", "AXIN2": "TissueArchitecture", "SMAD2": "TissueArchitecture", "SMAD3": "TissueArchitecture",
    "SMAD4": "TissueArchitecture", "TGFBR1": "TissueArchitecture", "TGFBR2": "TissueArchitecture", "NOTCH1": "TissueArchitecture",
    "NOTCH2": "TissueArchitecture", "NOTCH3": "TissueArchitecture", "NOTCH4": "TissueArchitecture", "FBXW7": "TissueArchitecture",
    "GJA1": "TissueArchitecture", "GJB2": "TissueArchitecture",

    # ChromatinRemodel — Chromatin Remodeling
    "KMT2D": "ChromatinRemodel", "KMT2C": "ChromatinRemodel", "KMT2A": "ChromatinRemodel", "KMT2B": "ChromatinRemodel",
    "SETD2": "ChromatinRemodel", "NSD1": "ChromatinRemodel", "CREBBP": "ChromatinRemodel", "EP300": "ChromatinRemodel",
    "ARID1A": "ChromatinRemodel", "ARID1B": "ChromatinRemodel", "ARID2": "ChromatinRemodel", "SMARCA4": "ChromatinRemodel",
    "KDM6A": "ChromatinRemodel", "BCOR": "ChromatinRemodel", "H3C7": "ChromatinRemodel", "ANKRD11": "ChromatinRemodel",

    # DNAMethylation — DNA Methylation
    "DNMT3A": "DNAMethylation", "DNMT3B": "DNAMethylation", "TET1": "DNAMethylation", "TET2": "DNAMethylation",
    "IDH1": "DNAMethylation", "IDH2": "DNAMethylation", "ATRX": "DNAMethylation", "DAXX": "DNAMethylation",

}

CHANNEL_NAMES = {
    "DDR": "DNA Damage Response",
    "CellCycle": "Cell Cycle / Apoptosis",
    "PI3K_Growth": "PI3K / Growth Signaling",
    "Endocrine": "Endocrine / Hormone",
    "Immune": "Immune Surveillance",
    "TissueArchitecture": "Tissue Architecture",
    "ChromatinRemodel": "Chromatin Remodeling",
    "DNAMethylation": "DNA Methylation",
}

CHANNEL_COLORS = {
    "DDR": "#e74c3c", "CellCycle": "#e67e22",
    "PI3K_Growth": "#f1c40f", "Endocrine": "#2ecc71",
    "Immune": "#3498db", "TissueArchitecture": "#9b59b6",
    "ChromatinRemodel": "#1abc9c", "DNAMethylation": "#e84393",
}

DRIVER_MUTATION_TYPES = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "Splice_Site", "In_Frame_Del", "In_Frame_Ins",
    "Nonstop_Mutation", "Translation_Start_Site",
}

NON_SILENT = DRIVER_MUTATION_TYPES  # alias for clarity

print(f"{len(CHANNEL_MAP)} genes mapped to {len(CHANNEL_NAMES)} channels.")

In [ ]:
# Cell 4: STRING PPI hub/leaf classification
#
# Fetch protein-protein interactions from STRING database.
# Compute within-channel degree. Top 3 per channel = hub, rest = leaf.

STRING_URL = "https://string-db.org/api/json/network"
SPECIES = 9606  # Homo sapiens
REQUIRED_SCORE = 700

def fetch_string_ppi(genes, species=SPECIES, required_score=REQUIRED_SCORE):
    """Fetch PPI from STRING in batches of 200 genes."""
    all_interactions = []
    gene_list = list(genes)

    for i in range(0, len(gene_list), 200):
        batch = gene_list[i:i+200]
        try:
            resp = requests.post(STRING_URL, data={
                "identifiers": "%0d".join(batch),
                "species": species,
                "required_score": required_score,
                "caller_identity": "paper7_reproduce",
            }, timeout=60)
            resp.raise_for_status()
            interactions = resp.json()
            all_interactions.extend(interactions)
            print(f"  STRING batch {i//200 + 1}: {len(interactions)} interactions")
            _time.sleep(1)  # rate limit courtesy
        except Exception as e:
            print(f"  STRING batch {i//200 + 1} failed: {e}")

    return all_interactions

print("Fetching STRING PPI data...")
all_genes = list(CHANNEL_MAP.keys())
ppi_data = fetch_string_ppi(all_genes)

# Parse interactions: extract gene symbols from preferredName fields
edges = []
for interaction in ppi_data:
    a = interaction.get("preferredName_A", "")
    b = interaction.get("preferredName_B", "")
    score = interaction.get("score", 0)
    if a in CHANNEL_MAP and b in CHANNEL_MAP and a != b:
        edges.append((a, b, score))

print(f"\nTotal edges among channel genes: {len(edges)}")

# Compute within-channel degree
within_ch_degree = {}
total_degree = {}

for a, b, score in edges:
    total_degree[a] = total_degree.get(a, 0) + 1
    total_degree[b] = total_degree.get(b, 0) + 1

    ch_a = CHANNEL_MAP.get(a)
    ch_b = CHANNEL_MAP.get(b)
    if ch_a == ch_b and ch_a is not None:
        within_ch_degree[(a, ch_a)] = within_ch_degree.get((a, ch_a), 0) + 1
        within_ch_degree[(b, ch_b)] = within_ch_degree.get((b, ch_b), 0) + 1

# Top 3 per channel = hub
HUB_GENES = {}
LEAF_GENES = {}

for ch in set(CHANNEL_MAP.values()):
    ch_genes = [(g, d) for (g, c), d in within_ch_degree.items() if c == ch]
    ch_genes.sort(key=lambda x: -x[1])
    HUB_GENES[ch] = set(g for g, d in ch_genes[:3])
    ch_all = {g for g, c in CHANNEL_MAP.items() if c == ch}
    LEAF_GENES[ch] = ch_all - HUB_GENES.get(ch, set())

GENE_POSITION = {}
GENE_DEGREE = {}
for ch, genes in HUB_GENES.items():
    for g in genes:
        GENE_POSITION[g] = "hub"
for ch, genes in LEAF_GENES.items():
    for g in genes:
        if g not in GENE_POSITION:
            GENE_POSITION[g] = "leaf"
for g in CHANNEL_MAP:
    GENE_DEGREE[g] = total_degree.get(g, 0)

# Print classification
print(f"\n{'Channel':25s}  {'Hubs (top 3 by within-ch degree)':40s}  {'Leaves':>6s}")
print("-" * 80)
for ch, name in CHANNEL_NAMES.items():
    hubs = HUB_GENES.get(ch, set())
    hub_str = ", ".join(f"{g}({within_ch_degree.get((g,ch),0)})" for g in sorted(hubs))
    n_leaf = len(LEAF_GENES.get(ch, set()))
    print(f"  {name:23s}  {hub_str:40s}  {n_leaf:>6d}")

In [ ]:
STUDY_ID = "msk_met_2021"
CACHE = "/content/"
os.makedirs(CACHE, exist_ok=True)


def fetch_clinical(study_id):
    path = os.path.join(CACHE, f"{study_id}_clinical.csv")
    if os.path.exists(path):
        print(f"  Loading cached clinical data...")
        return pd.read_csv(path, index_col=0)
    print(f"  Fetching clinical data for {study_id}...")
    data = api_get(f"/studies/{study_id}/clinical-data",
                   {"clinicalDataType": "PATIENT", "projection": "DETAILED"})
    df = pd.DataFrame(data)
    pivot = df.pivot_table(index="patientId", columns="clinicalAttributeId",
                           values="value", aggfunc="first")
    pivot.to_csv(path)
    print(f"  Cached {len(pivot)} patients.")
    return pivot


def fetch_mutations(study_id):
    path = os.path.join(CACHE, f"{study_id}_mutations.csv")
    if os.path.exists(path):
        print(f"  Loading cached mutations...")
        return pd.read_csv(path)
    print(f"  Fetching mutations for {study_id}...")
    samples = api_get(f"/studies/{study_id}/samples")
    sample_ids = [s["sampleId"] for s in samples]
    profiles = api_get(f"/studies/{study_id}/molecular-profiles")
    mut_profile = next(
        (p["molecularProfileId"] for p in profiles
         if p.get("molecularAlterationType") == "MUTATION_EXTENDED"), None)
    if not mut_profile:
        raise ValueError(f"No mutation profile for {study_id}")
    all_muts = []
    batch_size = 500
    for i in range(0, len(sample_ids), batch_size):
        batch = sample_ids[i:i + batch_size]
        muts = api_post(f"/molecular-profiles/{mut_profile}/mutations/fetch",
                        {"sampleIds": batch})
        all_muts.extend(muts)
        if (i // batch_size) % 20 == 0:
            print(f"    batch {i // batch_size + 1}/{(len(sample_ids) - 1) // batch_size + 1}")
    df = pd.json_normalize(all_muts)
    if "gene.hugoGeneSymbol" not in df.columns and "keyword" in df.columns:
        df["gene.hugoGeneSymbol"] = df["keyword"].apply(
            lambda k: str(k).split()[0] if pd.notna(k) else None)
    df.to_csv(path, index=False)
    print(f"  Cached {len(df)} mutations.")
    return df


print(f"=== Fetching {STUDY_ID} ===")
clinical_df = fetch_clinical(STUDY_ID)
mut_df = fetch_mutations(STUDY_ID)
print(f"Clinical: {len(clinical_df)} patients, Mutations: {len(mut_df)} rows")

In [ ]:
# Cell 6: Build patient dataframe with hub/leaf classification

gene_col = "gene.hugoGeneSymbol"

# Filter to non-silent, channel-mapped, position-classified mutations
mut = mut_df[mut_df["mutationType"].isin(NON_SILENT)].copy()
mut["channel"] = mut[gene_col].map(CHANNEL_MAP)
mut["position"] = mut[gene_col].map(GENE_POSITION)
mut["degree"] = mut[gene_col].map(GENE_DEGREE).fillna(0)
mut = mut.dropna(subset=["channel", "position"])

# Per patient-channel: has_hub, has_leaf
pcp = (
    mut.groupby(["patientId", "channel"])["position"]
    .agg(lambda x: set(x))
    .reset_index()
)
pcp["has_hub"] = pcp["position"].apply(lambda s: "hub" in s)
pcp["has_leaf"] = pcp["position"].apply(lambda s: "leaf" in s)

# Per patient summary
psumm = pcp.groupby("patientId").agg(
    channel_count=("channel", "nunique"),
    any_hub=("has_hub", "any"),
    any_leaf=("has_leaf", "any"),
).reset_index()
psumm["max_position"] = psumm["any_hub"].apply(lambda x: "hub" if x else "leaf")

# Merge with survival
surv = clinical_df[["OS_STATUS", "OS_MONTHS"]].copy().reset_index()
surv.columns = ["patientId", "os_status", "os_months"]
surv["os_months"] = pd.to_numeric(surv["os_months"], errors="coerce")
surv["event"] = surv["os_status"].apply(lambda x: 1 if "DECEASED" in str(x) else 0)
surv = surv.dropna(subset=["os_months"])
surv = surv[surv["os_months"] > 0]

df = psumm.merge(surv[["patientId", "os_months", "event"]], on="patientId", how="inner")

print(f"Patients with classified mutations: {len(df):,}")
print(f"  Hub (any hub mutation): {(df['max_position']=='hub').sum():,}")
print(f"  Leaf (leaf-only): {(df['max_position']=='leaf').sum():,}")

In [ ]:
# Cell 7: Table 1 — Within-channel hub vs leaf survival

print("=== TABLE 1: WITHIN-CHANNEL HUB VS LEAF ===\n")
print(f"{'Channel':20s}  {'Hub n':>7s}  {'Leaf n':>7s}  {'HR':>7s}  {'95% CI':>18s}  {'p':>10s}")
print("-" * 75)

for ch_name in ["PI3K_Growth", "CellCycle", "DDR"]:
    ch_data = pcp[pcp["channel"] == ch_name].copy()
    hub_pts = ch_data[ch_data["has_hub"] & ~ch_data["has_leaf"]]["patientId"]
    leaf_pts = ch_data[~ch_data["has_hub"] & ch_data["has_leaf"]]["patientId"]

    ch_surv = pd.DataFrame({
        "patientId": pd.concat([hub_pts, leaf_pts]),
        "hub_flag": [1]*len(hub_pts) + [0]*len(leaf_pts),
    })
    ch_surv = ch_surv.merge(surv[["patientId", "os_months", "event"]], on="patientId", how="inner")

    n_hub = (ch_surv["hub_flag"] == 1).sum()
    n_leaf = (ch_surv["hub_flag"] == 0).sum()

    if n_hub >= 20 and n_leaf >= 20:
        try:
            cph = CoxPHFitter()
            cph.fit(ch_surv[["os_months", "event", "hub_flag"]],
                    duration_col="os_months", event_col="event")
            r = cph.summary.loc["hub_flag"]
            hr = r["exp(coef)"]
            lo = r["exp(coef) lower 95%"]
            hi = r["exp(coef) upper 95%"]
            p = r["p"]
            print(f"  {CHANNEL_NAMES[ch_name]:18s}  {n_hub:>7,d}  {n_leaf:>7,d}  "
                  f"{hr:>7.3f}  ({lo:.3f}–{hi:.3f})  {p:>10.2e}")
        except Exception as e:
            print(f"  {CHANNEL_NAMES[ch_name]:18s}  {n_hub:>7,d}  {n_leaf:>7,d}  Cox failed: {e}")
    else:
        print(f"  {CHANNEL_NAMES[ch_name]:18s}  {n_hub:>7,d}  {n_leaf:>7,d}  Insufficient data")

print()
print("Key: HR > 1 = hub mutations carry worse prognosis.")
print("Bottleneck channels (PI3K, CellCycle) show effect; parallel channel (DDR) does not.")

In [ ]:
# Cell 8: Multivariate Cox — OS ~ hub_any + channel_count

df_mv = df[["os_months", "event", "max_position", "channel_count"]].copy()
df_mv["hub_any"] = (df_mv["max_position"] == "hub").astype(int)
df_mv = df_mv.drop(columns=["max_position"])

print("=== MULTIVARIATE COX: hub_any + channel_count ===\n")
print(f"N = {len(df_mv):,}")

cph = CoxPHFitter()
cph.fit(df_mv[["os_months", "event", "hub_any", "channel_count"]],
        duration_col="os_months", event_col="event")

for var in ["hub_any", "channel_count"]:
    r = cph.summary.loc[var]
    print(f"  {var:15s}: HR={r['exp(coef)']:.3f} "
          f"(95% CI {r['exp(coef) lower 95%']:.3f}–{r['exp(coef) upper 95%']:.3f}), "
          f"p={r['p']:.2e}")

print(f"\nC-index: {cph.concordance_index_:.4f}")
print("\nHub status is significant even after controlling for number of channels.")

In [ ]:
# Cell 9: TP53/KRAS sensitivity analysis

tp53_kras = {"TP53", "KRAS"}

# Patient max degree (all genes)
mut_with_degree = mut.copy()
pat_degree_all = mut_with_degree.groupby("patientId")["degree"].max().reset_index()
pat_degree_all.columns = ["patientId", "max_degree"]

# Patient max degree EXCLUDING TP53/KRAS
mut_no_tk = mut_with_degree[~mut_with_degree[gene_col].isin(tp53_kras)]
pat_degree_notk = mut_no_tk.groupby("patientId")["degree"].max().reset_index()
pat_degree_notk.columns = ["patientId", "max_degree_no_tk"]

deg_df = surv[["patientId", "os_months", "event"]].copy()
deg_df = deg_df.merge(pat_degree_all, on="patientId", how="inner")
deg_df = deg_df.merge(pat_degree_notk, on="patientId", how="left")
deg_df = deg_df.merge(psumm[["patientId", "channel_count"]], on="patientId", how="left")
deg_df["channel_count"] = deg_df["channel_count"].fillna(0)

print("=== TP53/KRAS SENSITIVITY ANALYSIS ===\n")

# 4a: All genes
print("4a. Continuous max degree (all genes):")
c_all = concordance_index(deg_df["os_months"], -deg_df["max_degree"], deg_df["event"])
print(f"  C-index: {c_all:.4f}")

cph = CoxPHFitter()
cox_in = deg_df[["os_months", "event", "max_degree", "channel_count"]].dropna()
cph.fit(cox_in, duration_col="os_months", event_col="event")
for var in ["max_degree", "channel_count"]:
    r = cph.summary.loc[var]
    print(f"  {var}: HR={r['exp(coef)']:.4f}, p={r['p']:.2e}")

# 4b: Excluding TP53/KRAS
print("\n4b. Continuous max degree (EXCLUDING TP53/KRAS):")
deg_notk = deg_df.dropna(subset=["max_degree_no_tk"])
if len(deg_notk) > 100:
    c_notk = concordance_index(deg_notk["os_months"], -deg_notk["max_degree_no_tk"], deg_notk["event"])
    print(f"  C-index: {c_notk:.4f}")
    print(f"  Signal {'REVERSES' if c_notk < 0.5 else 'persists'} without TP53/KRAS")

    cph = CoxPHFitter()
    cox_in = deg_notk[["os_months", "event", "max_degree_no_tk", "channel_count"]].dropna()
    cph.fit(cox_in, duration_col="os_months", event_col="event")
    for var in ["max_degree_no_tk", "channel_count"]:
        r = cph.summary.loc[var]
        print(f"  {var}: HR={r['exp(coef)']:.4f}, p={r['p']:.2e}")

# 4c: Binary hub/leaf excluding TP53/KRAS-only patients
print("\n4c. Binary hub/leaf (excluding TP53/KRAS-only hub patients):")
hub_genes_per_pat = mut[mut["position"] == "hub"].groupby("patientId")[gene_col].apply(set).reset_index()
hub_genes_per_pat.columns = ["patientId", "hub_genes"]
hub_genes_per_pat["has_other_hub"] = hub_genes_per_pat["hub_genes"].apply(lambda s: len(s - tp53_kras) > 0)

leaf_patients = set(psumm[psumm["max_position"] == "leaf"]["patientId"])
other_hub_patients = set(hub_genes_per_pat[hub_genes_per_pat["has_other_hub"]]["patientId"])

df_notk = df[df["patientId"].isin(leaf_patients | other_hub_patients)].copy()
df_notk["hub_any"] = df_notk["patientId"].isin(other_hub_patients).astype(int)

n_hub = df_notk["hub_any"].sum()
n_leaf = (df_notk["hub_any"] == 0).sum()
print(f"  Hub (non-TP53/KRAS): {n_hub:,}  |  Leaf: {n_leaf:,}")

if n_hub >= 20 and n_leaf >= 20:
    cph = CoxPHFitter()
    cox_in = df_notk[["os_months", "event", "hub_any", "channel_count"]].dropna()
    cph.fit(cox_in, duration_col="os_months", event_col="event")
    for var in ["hub_any", "channel_count"]:
        r = cph.summary.loc[var]
        print(f"  {var}: HR={r['exp(coef)']:.3f} "
              f"({r['exp(coef) lower 95%']:.3f}–{r['exp(coef) upper 95%']:.3f}), p={r['p']:.2e}")

In [ ]:
# Cell 10: Figure 1 — KM curves: hub vs leaf (all patients)

hub_grp = df[df["max_position"] == "hub"]
leaf_grp = df[df["max_position"] == "leaf"]

lr = logrank_test(hub_grp["os_months"], leaf_grp["os_months"],
                  hub_grp["event"], leaf_grp["event"])

fig, ax = plt.subplots(figsize=(8, 5))
kmf = KaplanMeierFitter()

kmf.fit(hub_grp["os_months"], hub_grp["event"], label=f"Hub (n={len(hub_grp):,})")
kmf.plot_survival_function(ax=ax, ci_show=True, color="#c0392b")

kmf.fit(leaf_grp["os_months"], leaf_grp["event"], label=f"Leaf (n={len(leaf_grp):,})")
kmf.plot_survival_function(ax=ax, ci_show=True, color="#2980b9")

ax.set_xlabel("Overall Survival (months)", fontsize=12)
ax.set_ylabel("Survival Probability", fontsize=12)
ax.set_title("Hub vs Leaf Mutation Survival\nMSK-MET 2021", fontsize=13, fontweight="bold")
ax.legend(fontsize=11, loc="lower left")
ax.annotate(f"Log-rank p = {lr.p_value:.2e}", xy=(0.98, 0.98),
            xycoords="axes fraction", ha="right", va="top", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.7))
ax.set_xlim(0, min(df["os_months"].quantile(0.95), 120))
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Figure 2 — Per-channel architecture comparison

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ch_name in zip(axes, ["PI3K_Growth", "CellCycle", "DDR"]):
    ch_data = pcp[pcp["channel"] == ch_name].copy()
    hub_pts = ch_data[ch_data["has_hub"] & ~ch_data["has_leaf"]]["patientId"]
    leaf_pts = ch_data[~ch_data["has_hub"] & ch_data["has_leaf"]]["patientId"]

    ch_surv = pd.DataFrame({
        "patientId": pd.concat([hub_pts, leaf_pts]),
        "hub_flag": [1]*len(hub_pts) + [0]*len(leaf_pts),
    })
    ch_surv = ch_surv.merge(surv[["patientId", "os_months", "event"]], on="patientId", how="inner")

    kmf = KaplanMeierFitter()
    h = ch_surv[ch_surv["hub_flag"] == 1]
    l = ch_surv[ch_surv["hub_flag"] == 0]

    if len(h) >= 10:
        kmf.fit(h["os_months"], h["event"], label=f"Hub (n={len(h):,})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color="#c0392b")
    if len(l) >= 10:
        kmf.fit(l["os_months"], l["event"], label=f"Leaf (n={len(l):,})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color="#2980b9")

    if len(h) >= 10 and len(l) >= 10:
        lr = logrank_test(h["os_months"], l["os_months"], h["event"], l["event"])
        try:
            cph = CoxPHFitter()
            cph.fit(ch_surv[["os_months", "event", "hub_flag"]],
                    duration_col="os_months", event_col="event")
            hr = cph.summary.loc["hub_flag"]["exp(coef)"]
            ax.text(0.95, 0.05, f"HR={hr:.2f}, p={lr.p_value:.2e}",
                    transform=ax.transAxes, ha="right", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        except Exception:
            pass

    arch = "bottleneck" if ch_name in ["PI3K_Growth", "CellCycle"] else "parallel"
    ax.set_title(f"{CHANNEL_NAMES[ch_name]}\n({arch} architecture)", fontweight="bold")
    ax.set_xlabel("Months")
    ax.set_ylabel("Overall Survival")
    ax.legend(fontsize=8, loc="lower left")
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()